# Decorators

## Definiton:

A decorator is a callable that takes a function (or another callable) and returns a callable with modified or extended behavior.

The core idea is:

```python
original -> decorator -> wrapped function
```

Python's `@name` syntax is used for applying a decorator immediately after a function is defined.


In [1]:
def shout(func):
    def wrapper():
        result = func()
        return result.upper()
    return wrapper

def greet():
    return "hello"

decorated_greet = shout(greet)

print(decorated_greet())

HELLO


## 2. The `@decorator` syntax

This:

```python
@shout
def greet():
    ...
```

is equivalent in effect to:

```python
def greet():
    ...

greet = shout(greet)
```

The decoration happens when the `def` statement is executed, not each time the function is called.


In [2]:
def shout(func):
    def wrapper():
        return func().upper()
    return wrapper

@shout
def greet():
    return "hello"

print(greet())

HELLO


## 3. Why `wrapper(*args, **kwargs)` matters

A reusable decorator should usually accept arbitrary arguments and forward them to the original function.

This lets the decorator work with functions having different signatures.


In [4]:
def log_call(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Returned {result!r}")
        return result
    return wrapper

@log_call
def add(a, b):
    return a + b

@log_call
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print(add(2, 3))
print(greet("Asha", greeting="Hi"))

Calling add
Returned 5
5
Calling greet
Returned 'Hi, Asha!'
Hi, Asha!


## 4. Decorators are built from closures

The wrapper function closes over `func`. That is why it can still access the original function after the outer decorator function has returned.

Mental model:

1. `log_call(add)` runs.
2. `func` refers to `add`.
3. `wrapper` is created.
4. `wrapper` remembers `func`.
5. `add` is rebound to `wrapper`.


## 5. Preserving metadata with `functools.wraps`

Without `wraps`, the wrapper can hide useful metadata such as `__name__` and `__doc__`.

`functools.wraps(original_function)` is a convenience decorator that uses `update_wrapper` to preserve important function metadata.


In [5]:
from functools import wraps

def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_call
def greet(name):
    """Return a greeting."""
    return f"Hello, {name}"

print(greet("Maya"))
print(greet.__name__)
print(greet.__doc__)

Calling greet
Hello, Maya
greet
Return a greeting.


## 6. Decorator with arguments

A decorator with configuration needs one extra layer.

Structure:

```python
def decorator_factory(option):
    def decorator(func):
        def wrapper(*args, **kwargs):
            ...
        return wrapper
    return decorator
```

Then:

```python
@decorator_factory(option)
def function():
    ...
```


In [6]:
from functools import wraps

def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say_hi(name):
    print(f"Hi, {name}!")

say_hi("Sam")

Hi, Sam!
Hi, Sam!
Hi, Sam!


## 7. Practical decorator: timing

A decorator can measure execution time without changing the business logic of the function.


In [1]:
from functools import wraps
from time import perf_counter

def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = perf_counter() - start
            print(f"{func.__name__}: {elapsed:.6f}s")
    return wrapper

@timed
def compute():
    return sum(i * i for i in range(100_000))

print(compute())

compute: 0.008785s
333328333350000


## 8. Practical decorator: validation

Decorators can enforce preconditions before calling a function.


In [2]:
from functools import wraps

def require_positive(func):
    @wraps(func)
    def wrapper(value, *args, **kwargs):
        if value <= 0:
            raise ValueError("value must be positive")
        return func(value, *args, **kwargs)
    return wrapper

@require_positive
def reciprocal(value):
    return 1 / value

print(reciprocal(4))

# reciprocal(-1)  # ValueError

0.25


## 9. Practical decorator: caching

Caching is a classic example of behavior that can be added without rewriting the original function.

For real code, prefer `functools.cache` or `functools.lru_cache` where appropriate instead of implementing a cache decorator yourself.


In [3]:
from functools import cache

@cache
def factorial(n):
    print("Computing", n)
    return 1 if n == 0 else n * factorial(n - 1)

print(factorial(5))
print(factorial(5))  # cached result

Computing 5
Computing 4
Computing 3
Computing 2
Computing 1
Computing 0
120
120


## 10. Stacking decorators

Decorators are applied bottom-up.

```python
@A
@B
def f():
    ...
```

is equivalent to:

```python
f = A(B(f))
```

Therefore the order can change behavior.


In [4]:
from functools import wraps

def add_prefix(prefix):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return prefix + func(*args, **kwargs)
        return wrapper
    return decorator

@add_prefix("[OUTER] ")
@add_prefix("[INNER] ")
def message():
    return "hello"

print(message())

[OUTER] [INNER] hello


## 11. Practice exercises

1. Write a `debug` decorator that prints arguments and the return value.
2. Write a `require_keyword` decorator that checks whether a keyword is present.
3. Write a decorator factory `repeat(n)`.

**Note:** The call flow

`decorator factory -> decorator -> wrapper -> original function`.


In [ ]:
#1
from functools import wraps
def debug(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result=func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper
@debug
def add(a,b):
    return a+b
print(add(3,4))

Calling add with args=(3, 4), kwargs={}
add returned 7
7


In [8]:
#2
from functools import wraps
def require_keyword(func):
    @wraps(func)
    def wrapper(*args,**kwargs):
        if not kwargs:
            raise TypeError(f"{func.__name__} requires keyword arguments")
        return func(*args,**kwargs)
    return wrapper
@require_keyword
def greet(name, message):
    return f"{message}, {name}!"
print(greet(name="Alice", message="Hello"))
# print(greet()) TypeError: greet requires keyword arguments

Hello, Alice!


In [14]:
#3
from functools import wraps
def repeat(n):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result=None
            for _ in range(n):
                result=func(*args,**kwargs)
            return result
        return wrapper
    return decorator
@repeat(5)
def print_message(msg):
    print(msg)

print_message("Hello!")


Hello!
Hello!
Hello!
Hello!
Hello!


In [ ]:
#4
